# 05 · Video tracking and counting crossings

**Level:** intermediate · **Presentation:** 35–50 minutes · **Prerequisite:** notebook 04.

Detecting a person in 30 frames does not mean 30 people visited. Here we separate detections,
tracker IDs, and **line-crossing events**. You will build a deterministic counter first, then
connect it to YOLO11n + ByteTrack on video. CPU, bounded loops, and inline Jupyter output are defaults.

The generated demo clip pans a single bus photograph. It is an **artificial repeated scene**
for exercising the pipeline, not real object motion, physical visitor counts, or a tracking
accuracy benchmark. For meaningful counting, use a fixed camera, an appropriate line, and
manually labeled real video. An ID is a temporary tracker label, not a verified identity.

**Presenter question to pose aloud:** “What exactly should we count: frames, track IDs, or events?”


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


In [ ]:
import os
import importlib.util
from dataclasses import dataclass, field
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from course_utils import make_demo_video, load_yolo

OFFLINE = os.getenv("COURSE_OFFLINE", "0") == "1"
SMOKE = os.getenv("COURSE_SMOKE", "0") == "1"
MODEL = "yolo11n.pt"
DEVICE = "cpu"
MAX_FRAMES = 8 if SMOKE else 60
SHOW_EVERY = 10
LINE_FRACTION = 0.40  # Count line at 40% of image width; choose for your own scene.
OUT = ROOT / "outputs" / "05_tracking"
OUT.mkdir(parents=True, exist_ok=True)


## 1. Define the event before writing the loop

Our vertical line has a small deadband. A track must be seen clearly on one side and later
clearly on the other. Centers inside the deadband preserve the last stable side. This prevents
tiny jitter near the line from generating an immediate event. We count each **ID + direction**
at most once per run. A first observation already on the right is not a crossing.

Long observation gaps reset the previous side, because a crossing during a long absence is
uncertain. Short gaps may still infer a crossing. This is an explicit policy, not perfect truth.


In [ ]:
@dataclass
class LineCounter:
    line_x: float
    margin: float = 8.0
    max_gap: int = 8
    state: dict = field(default_factory=dict)
    counted: set = field(default_factory=set)
    events: list = field(default_factory=list)

    def update(self, track_id, x_center, frame_index):
        track_id = int(track_id)
        old_side, last_frame = self.state.get(track_id, (None, frame_index))
        if frame_index - last_frame > self.max_gap:
            old_side = None
        side = (-1 if x_center < self.line_x - self.margin else
                1 if x_center > self.line_x + self.margin else 0)
        if side == 0:
            self.state[track_id] = (old_side, frame_index)
            return None
        event = None
        if old_side is not None and side != old_side:
            direction = "left_to_right" if side == 1 else "right_to_left"
            key = (track_id, direction)
            if key not in self.counted:
                self.counted.add(key)
                event = {"frame": int(frame_index), "track_id": track_id,
                         "direction": direction, "x_center": float(x_center)}
                self.events.append(event)
        self.state[track_id] = (side, frame_index)
        return event

    @property
    def totals(self):
        tally = Counter(event["direction"] for event in self.events)
        return {"left_to_right": tally["left_to_right"],
                "right_to_left": tally["right_to_left"]}


## 2. Test the counting rule without a model or internet

These are **known synthetic center trajectories**, not detector outputs. ID 1 moves right,
ID 2 moves left, and ID 3 jitters in the deadband. Expect exactly one crossing each way.
Keeping the event logic separate makes it possible to debug counting independently of detection.


In [ ]:
trajectories = {
    1: [20, 30, 40, 47, 50, 54, 62, 72, 80],
    2: [82, 75, 68, 60, 53, 48, 39, 30, 22],
    3: [50, 49, 51, 50, 52, 48, 49, 51, 50],
}
demo_counter = LineCounter(line_x=50, margin=5)
for frame_index in range(9):
    for track_id, positions in trajectories.items():
        demo_counter.update(track_id, positions[frame_index], frame_index)
display(pd.DataFrame(demo_counter.events))
print(demo_counter.totals)
assert demo_counter.totals == {"left_to_right": 1, "right_to_left": 1}

fig, ax = plt.subplots(figsize=(9, 4))
for track_id, positions in trajectories.items():
    ax.plot(positions, marker="o", label=f"known ID {track_id}")
ax.axhspan(45, 55, alpha=0.15, color="gray", label="deadband")
ax.axhline(50, color="black", linestyle="--")
ax.set(xlabel="Frame index", ylabel="Horizontal center (pixels)",
       title="Event logic can be verified before running a detector")
ax.legend()
plt.show()


## 3. Prepare a short video and check tracking dependencies

`make_demo_video()` uses the cached/downloaded official sample image and creates a short
artificial pan locally. Set `VIDEO_PATH` to your own local file for real footage.
The line at 40% of image width intersects a person's image trajectory in the artificial pan;
moving the line to the center may produce no events. Choose placement to fit the real scene.
Only one source runs at a time. A fresh model is created per source so old track IDs do not
leak from one clip into another. ByteTrack's assignment solver needs the `lap` package.


In [ ]:
VIDEO_PATH = None  # Example: ROOT / "assets" / "my_fixed_camera_clip.mp4"
tracking_ready = not OFFLINE
if OFFLINE:
    print("Offline mode: real video/model section skipped; deterministic counter above still runs.")
elif importlib.util.find_spec("lap") is None:
    tracking_ready = False
    print("Tracking needs lap. Install the root requirements.txt in this kernel, restart, and rerun.")
else:
    import cv2
    VIDEO_PATH = Path(VIDEO_PATH) if VIDEO_PATH is not None else make_demo_video()
    if not VIDEO_PATH.exists():
        raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")
    print("Video:", VIDEO_PATH)


## 4. Track consecutive frames and update the counter

`persist=True` preserves state across consecutive frames. `classes=[person_id]` limits this
demonstration to people. We process every frame up to the bound; the display cadence only
changes rendering, not tracking. The `finally` block releases the camera/file even after an error.

Inline previews show the latest processed frame. They are not a real-time playback guarantee.
We retain a small table of track centers, a separate event table, and one final annotated frame.


In [ ]:
def run_tracking(source, max_frames=60, show_every=10, line_fraction=0.5):
    if max_frames < 1 or show_every < 1 or not 0 < line_fraction < 1:
        raise ValueError("Use positive frame/display bounds and line_fraction between 0 and 1.")
    import cv2
    detector = load_yolo(MODEL)  # New instance = new tracker state.
    person_ids = [k for k, name in detector.names.items() if name == "person"]
    if not person_ids:
        raise ValueError("This model has no 'person' class; choose your target class explicitly.")
    capture = cv2.VideoCapture(source if isinstance(source, int) else str(source))
    rows, counter, handle, last_rgb = [], None, None, None
    try:
        if not capture.isOpened():
            raise RuntimeError(f"Cannot open {source}. Check file/codec or Windows camera permission.")
        for frame_index in range(max_frames):
            ok, frame = capture.read()
            if not ok:
                break
            height, width = frame.shape[:2]
            if counter is None:
                counter = LineCounter(line_x=width * line_fraction,
                                      margin=max(5, width * 0.015), max_gap=8)
            prediction = detector.track(frame, persist=True, tracker="bytetrack.yaml",
                                        classes=person_ids, imgsz=320, conf=0.10,
                                        iou=0.70, device=DEVICE, verbose=False)[0]
            boxes = prediction.boxes
            if boxes is not None and boxes.id is not None:
                for box, track_id in zip(boxes.xyxy.cpu().numpy(),
                                          boxes.id.int().cpu().tolist()):
                    center_x = float((box[0] + box[2]) / 2)
                    center_y = float((box[1] + box[3]) / 2)
                    event = counter.update(track_id, center_x, frame_index)
                    rows.append({"frame": frame_index, "track_id": track_id,
                                 "center_x": center_x, "center_y": center_y,
                                 "crossing": event["direction"] if event else ""})
            annotated = prediction.plot()
            x = int(counter.line_x)
            cv2.line(annotated, (x, 0), (x, height - 1), (0, 255, 255), 2)
            caption = f"Frame {frame_index} | L->R {counter.totals['left_to_right']} | R->L {counter.totals['right_to_left']}"
            cv2.putText(annotated, caption, (10, 25), cv2.FONT_HERSHEY_SIMPLEX,
                        0.55, (0, 0, 255), 2)
            last_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            if frame_index % show_every == 0 or frame_index == max_frames - 1:
                preview = Image.fromarray(last_rgb)
                preview.thumbnail((700, 500))
                if handle is None:
                    handle = display(preview, display_id=True)
                else:
                    handle.update(preview)
    finally:
        capture.release()
    track_table = pd.DataFrame(rows, columns=["frame", "track_id", "center_x", "center_y", "crossing"])
    events = pd.DataFrame(counter.events if counter else [],
                          columns=["frame", "track_id", "direction", "x_center"])
    if last_rgb is None:
        raise RuntimeError("The source opened but returned no readable frames. Try a different local video/codec.")
    return track_table, events, last_rgb


In [ ]:
tracks = pd.DataFrame()
events = pd.DataFrame()
if tracking_ready:
    tracks, events, last_frame = run_tracking(VIDEO_PATH, MAX_FRAMES, SHOW_EVERY,
                                             line_fraction=LINE_FRACTION)
    tracks.to_csv(OUT / "track_centers.csv", index=False)
    events.to_csv(OUT / "crossing_events.csv", index=False)
    Image.fromarray(last_frame).save(OUT / "last_frame.png")
    print("Track observations (NOT visitors):", len(tracks))
    print("Distinct tracker IDs (NOT verified people):", tracks["track_id"].nunique())
    print("Line-crossing events:", len(events))
    display(events)
    print("Saved artifacts to", OUT)


## 5. Inspect the temporal behavior

A broken line or a new color can reveal track fragmentation. A plausible drawing alone
does not validate ID accuracy. **Zero crossings is a valid result**, especially in a short
artificial clip or smoke run; use the deterministic example to demonstrate an expected event.


In [ ]:
if not tracks.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    for track_id, group in tracks.groupby("track_id"):
        ax.plot(group["frame"], group["center_x"], label=f"ID {track_id}")
    ax.axhline(last_frame.shape[1] * LINE_FRACTION, color="black", linestyle="--", label="count line")
    ax.set(xlabel="Frame", ylabel="Center x (pixels)", title="Observed tracker trajectories")
    ax.legend(ncol=3)
    plt.show()
else:
    print("No tracked observations available for plotting.")


## 6. Optional live camera — explicit opt-in

Set `RUN_WEBCAM=True` only when you intend to open your camera. The loop is bounded and releases
the device. The camera remains local; the preview is shown in the notebook. For a presentation,
use a local recorded clip first so lighting, permissions, and hardware do not disrupt the lesson.
The cell below does not save camera frames to disk.


In [ ]:
RUN_WEBCAM = False
WEBCAM_INDEX = 0
WEBCAM_MAX_FRAMES = 100
if RUN_WEBCAM and tracking_ready:
    webcam_tracks, webcam_events, webcam_last_frame = run_tracking(
        WEBCAM_INDEX, max_frames=WEBCAM_MAX_FRAMES, show_every=5)
    display(webcam_events)
else:
    print("Camera not opened. Enable RUN_WEBCAM and the real-model section to try it.")


## 7. Exercises and solutions

1. Feed an ID that appears at x=80 first. Should it count as left-to-right? **No:** previous-side
   evidence is missing.
2. Let the same ID cross right, left, then right again. Our declared policy records two events
   total, one per direction. To count every trip, remove the `counted` de-duplication and add
   a suitable cooldown; report the policy with the result.
3. Increase the deadband and describe the tradeoff: fewer jitter events, but a short crossing
   may not reach the opposite stable side.
4. Use a fixed-camera clip and manually label crossings. Compare event direction/time and
   missed events. Counting errors may arise from detector misses, ID switches, line placement,
   or the event policy.

Run these edge cases to make the counting policy concrete:


In [ ]:
policy_test = LineCounter(50, margin=5)
for frame, x in enumerate([20, 80, 20, 80, 80]):
    policy_test.update(7, x, frame)
assert policy_test.totals == {"left_to_right": 1, "right_to_left": 1}
first_seen_right = LineCounter(50, margin=5)
first_seen_right.update(9, 80, 0)
assert len(first_seen_right.events) == 0
gap_test = LineCounter(50, margin=5, max_gap=3)
gap_test.update(2, 20, 0)
gap_test.update(2, 80, 10)
assert len(gap_test.events) == 0
print("Verified: de-duplication, first observation, and long-gap policies.")


**Presenter checkpoint:** explain why `len(tracks)`, distinct IDs, and `len(events)` are different
quantities. A tracker can assign new IDs after occlusion, so even de-duplicated ID counts can
overcount physical objects. This short demo keeps state in memory; a long-running service needs
state expiry, logs, and a tested policy for camera interruptions.

Primary references: [Ultralytics tracking and persistent frames](https://docs.ultralytics.com/modes/track/),
[ByteTrack implementation and paper](https://github.com/ifzhang/ByteTrack).
**Next:** custom-data preparation and training before the advanced deployment lessons.
